# Merge 4 Fine-tuned Llama 3.1 8B Models into Mixtral-style MoE

This notebook merges 4 individually fine-tuned Llama 3.1 8B models into a Mixture of Experts using mergekit-moe.


## 1. Installation

In [ ]:
!pip install -qqq mergekit torch transformers

## 2. Configuration

In [ ]:
import os
from pathlib import Path

# Expert model paths - your fine-tuned Llama 3.1 8B models
EXPERT_PATHS = [
    "./output_moe/llama3_expert_1",
    "./output_moe/llama3_expert_2",
    "./output_moe/llama3_expert_3",
    "./output_moe/llama3_expert_4",
]

OUTPUT_DIR = "./output_merged_moe"
CONFIG_FILE = "merge_moe_config.yaml"

print("Configuration set!")

## 3. Verify Expert Models

In [ ]:
# Check that all expert models exist
for expert_path in EXPERT_PATHS:
    if Path(expert_path).exists():
        print(f"✓ Found: {expert_path}")
    else:
        print(f"✗ Missing: {expert_path}")

print("\nAll experts verified!")

## 4. Create MoE Merge Configuration

Gate modes:
- **hidden**: Best quality, uses hidden states from prompts (default)
- **cheap_embed**: Low VRAM, uses token embeddings only
- **random**: For further training, randomly initializes gates

In [ ]:
import yaml

# MoE merge configuration
moe_config = {
    "base_model": EXPERT_PATHS[0],  # Use first expert as base for attention layers
    "gate_mode": "hidden",  # Best quality; alternatives: cheap_embed, random
    "dtype": "float16",
    "experts_per_token": 2,  # Each token routes to 2 experts
    "experts": [
        {
            "source_model": EXPERT_PATHS[0],
            "positive_prompts": [
                "What is machine learning and how does it work?",
                "Explain artificial intelligence in detail"
            ]
        },
        {
            "source_model": EXPERT_PATHS[1],
            "positive_prompts": [
                "Write a Python function to calculate fibonacci",
                "How do I debug this code error?"
            ]
        },
        {
            "source_model": EXPERT_PATHS[2],
            "positive_prompts": [
                "Explain quantum computing principles",
                "What is deep learning architecture?"
            ]
        },
        {
            "source_model": EXPERT_PATHS[3],
            "positive_prompts": [
                "Tell me a story about adventure",
                "How should I approach this problem?"
            ]
        }
    ]
}

# Save configuration
with open(CONFIG_FILE, 'w') as f:
    yaml.dump(moe_config, f, default_flow_style=False)

print(f"✓ Configuration saved to {CONFIG_FILE}")
print("\nConfiguration:")
print(yaml.dump(moe_config, default_flow_style=False))

## 5. Run MoE Merge

In [ ]:
import subprocess

# Run mergekit-moe
cmd = [
    "mergekit-moe",
    CONFIG_FILE,
    OUTPUT_DIR,
    "--cuda",
    "--lazy-unpickle"
]

print(f"Running: {' '.join(cmd)}")
print("This may take 30-60 minutes...\n")

result = subprocess.run(cmd, capture_output=False)

if result.returncode == 0:
    print("\n✓ Merge completed successfully!")
    print(f"✓ MoE model saved to: {OUTPUT_DIR}")
else:
    print(f"\n✗ Merge failed with error code: {result.returncode}")

## 6. Verify Merged Model

In [ ]:
from pathlib import Path

# Check output directory
output_path = Path(OUTPUT_DIR)

if output_path.exists():
    files = list(output_path.glob('*'))
    print(f"✓ Output directory exists: {OUTPUT_DIR}")
    print(f"✓ Contains {len(files)} files/directories:")
    for f in sorted(files)[:10]:
        print(f"  - {f.name}")
else:
    print(f"✗ Output directory not found: {OUTPUT_DIR}")

## 7. Next Steps

Your 4x8B MoE model is ready! You can now:

1. **Fine-tune the MoE**: Use `finetune_moe.py` to further train
2. **Push to HuggingFace**: Upload to the Hub for sharing
3. **Run inference**: Use the merged model directly

Model location: `./output_merged_moe`

In [ ]:
# Optional: Test inference
# from transformers import AutoTokenizer, AutoModelForCausalLM
# 
# tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
# model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR, device_map="auto")
# 
# prompt = "What is machine learning?"
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(**inputs, max_length=100)
# print(tokenizer.decode(outputs[0]))